[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/10-engineering-toolbox/03_code-quality.ipynb)

*Engineering Toolbox · Notebook 3 of 8*

# Code Quality, Linting & Type Hints

Build the checks that linters and type checkers run, with Python's ast module: unused imports, complexity, long lines, naming, mutable defaults, swallowed errors, docstring and type-hint coverage, runtime type checks, import sorting, duplicates, dead code, magic numbers, long functions, whitespace, shadowed builtins, old-style formatting, line counts, layer rules and rule selection.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A linter is a spell-checker for code: it reads without running anything and underlines the likely mistakes.

**What interviewers really test here:** Linters and formatters, type hints vs runtime validation, mutable defaults, bare except, complexity and how to roll rules out to an old codebase.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why code quality tools exist *(Basics)*
- [ ] 2. How a linter reads your code *(Core)*
- [ ] 3. Type hints and type checking *(Core)*
- [ ] 4. Keeping a codebase healthy *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Common rule families

| Prefix | From | Catches |
| --- | --- | --- |
| E / W | pycodestyle | style and whitespace |
| F | pyflakes | unused imports, undefined names |
| I | isort | import order |
| B | flake8-bugbear | likely bugs (mutable defaults, loop variable capture) |
| UP | pyupgrade | old syntax that has a modern form |
| S | bandit | security issues |

---
# Part 2: Lessons, step by step

## Lesson 1: Why code quality tools exist

*Level: Basics*

Code is read far more often than it's written. Teams agree on a **style** (PEP 8, enforced by a formatter so nobody argues about it) and run a **linter** that finds likely bugs: unused imports, undefined names, mutable defaults, bare `except`. Reviewers can then spend their time on design instead of commas.



```python
pip install ruff
ruff check .          # find problems
ruff check --fix .    # fix the safe ones automatically
ruff format .         # format the code

# pyproject.toml
[tool.ruff]
line-length = 88
[tool.ruff.lint]
select = ["E", "F", "I", "B", "UP"]   # pycodestyle, pyflakes, isort, bugbear, pyupgrade
ignore = ["E501"]
```

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q9](#q9), [Q2](#q2), [Q17](#q17), [Q20](#q20), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 2: How a linter reads your code

*Level: Core*

Linters don't run your program. They parse it into an **abstract syntax tree** with `ast.parse` and look for patterns: an import whose name never appears, a default argument that's a list, an `except` whose body is just `pass`. You can write a custom check for your team in a few lines.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
import ast


def ast_demo():
    tree = ast.parse("import os\ndef add(item, items=[]):\n    items.append(item)\n    return items\n")
    print(ast.dump(tree.body[1].args.defaults[0]))
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            print("function", node.name, "on line", node.lineno)


ast_demo()

List(ctx=Load())
function add on line 2


**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q3](#q3), [Q15](#q15), [Q6](#q6), [Q10](#q10), [Q11](#q11), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 3: Type hints and type checking

*Level: Core*

Type hints document what a function expects and returns, power editor autocompletion, and let **mypy** or **pyright** catch bugs before running anything. They're ignored at runtime — use **pydantic** when you need to *validate* data at the edges (API requests, config files).



```python
from dataclasses import dataclass

@dataclass
class Order:
    id: int
    amount: float
    note: str | None = None
    tags: list[str] = field(default_factory=list)

def total(orders: list[Order]) -> float:
    return sum(o.amount for o in orders)

total(["12.5"])   # mypy: Argument 1 has incompatible type "list[str]"

# runtime validation at the boundary
from pydantic import BaseModel
class OrderIn(BaseModel):
    id: int
    amount: float
OrderIn(id="7", amount="9.5")   # coerces to id=7, amount=9.5; invalid input raises
```

**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q16](#q16), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 4: Keeping a codebase healthy

*Level: Advanced*

- **Complexity**: functions with many branches are hard to test — split them. Aim for cyclomatic complexity under ~10 and functions that fit on a screen.
- **Duplication**: extract repeated blocks into functions (but don't abstract too early — "rule of three").
- **Architecture rules**: enforce that low-level modules never import high-level ones, so the code stays layered.
- Run all of it automatically with **pre-commit** locally and again in **CI**, so it never depends on memory.



```python
# .pre-commit-config.yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.8.0
    hooks: [{id: ruff, args: [--fix]}, {id: ruff-format}]
  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.13.0
    hooks: [{id: mypy}]
```

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q8](#q8), [Q18](#q18), [Q19](#q19). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
import ast

from collections import Counter, defaultdict


<a id="q1"></a>
### Q1. Lines Too Long

🟢 Warm-up · *E501*

Return `(line number, length)` for every line longer than `limit` characters (Black's default is 88).

**Examples**

- `src = "def f():\n    return 1\nx = 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'\n# bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\ny = 2\n", limit = 88` → `[(3, 96)]`

In [ ]:
def long_lines(src, limit):
    """
    Return (line number, length) for every line longer than limit characters
    (Black's default is 88).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(long_lines, [
    (("def f():\n    return 1\nx = 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa'\n# bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\ny = 2\n", 88), [(3, 96)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with enumerate.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with enumerate.

```python
def long_lines(src, limit):
    out = []
    n = 0
    for line in src.split("\n"):
        n += 1
        if len(line) > limit:
            out.append((n, len(line)))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def long_lines(src, limit):
    return [(i, len(l)) for i, l in enumerate(src.splitlines(), 1) if len(l) > limit]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with enumerate. That costs O(n) time and O(n) space."

"I can do better: A comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/lines-too-long)

<a id="q2"></a>
### Q2. Check Naming Conventions

🟢 Warm-up · *PEP 8 names*

Report `(line, name, rule)` for: functions not in `snake_case` (rule `'function'`), classes not in `CapWords` (`'class'`), and module-level assignments to names that are neither UPPER\_CASE constants nor snake\_case (`'variable'`). Use the regexes `SNAKE`, `CAPWORDS`, `UPPER`.

**Examples**

- `src = 'MAX_RETRIES = 3\nmaxSize = 10\n\nclass user_account:\n    pass\n\nclass HttpClient:\n    def SendRequest(self):\n        pass\n\ndef parse_config():\n    pass\n\ndef LoadData():\n    pass\n'` → `[(2, 'maxSize', 'variable'), (4, 'user_account', 'class'), (8, 'SendRequest', 'function'), (14, 'LoadData', 'function')]`

In [ ]:
import re

SNAKE = re.compile(r"^_{0,2}[a-z][a-z0-9_]*$")

CAPWORDS = re.compile(r"^_?[A-Z][a-zA-Z0-9]*$")

UPPER = re.compile(r"^_?[A-Z][A-Z0-9_]*$")

def naming(src):
    """
    Report (line, name, rule) for: functions not in snake_case (rule
    'function'), classes not in CapWords ('class'), and module-level assignments
    to names that are neither UPPER_CASE constants nor snake_case ('variable').
    Use the regexes SNAKE, CAPWORDS, UPPER.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(naming, [
    (('MAX_RETRIES = 3\nmaxSize = 10\n\nclass user_account:\n    pass\n\nclass HttpClient:\n    def SendRequest(self):\n        pass\n\ndef parse_config():\n    pass\n\ndef LoadData():\n    pass\n',), [(2, 'maxSize', 'variable'), (4, 'user_account', 'class'), (8, 'SendRequest', 'function'), (14, 'LoadData', 'function')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the tree; check each kind with if/elif.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A rule table of (node type, regex, rule).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk the tree; check each kind with if/elif.

```python
def naming(src):
    tree = ast.parse(src)
    out = []
    for n in ast.walk(tree):
        if isinstance(n, ast.FunctionDef) and not SNAKE.match(n.name):
            out.append((n.lineno, n.name, "function"))
        elif isinstance(n, ast.ClassDef) and not CAPWORDS.match(n.name):
            out.append((n.lineno, n.name, "class"))
    for n in tree.body:
        if isinstance(n, ast.Assign):
            for t in n.targets:
                if isinstance(t, ast.Name) and not (UPPER.match(t.id) or SNAKE.match(t.id)):
                    out.append((n.lineno, t.id, "variable"))
    return sorted(out)
```

**Cost:** O(n) time, O(v) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A rule table of (node type, regex, rule).

```python
def naming(src):
    tree = ast.parse(src)
    out = [(n.lineno, n.name, "function") for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and not SNAKE.match(n.name)]
    out += [(n.lineno, n.name, "class") for n in ast.walk(tree) if isinstance(n, ast.ClassDef) and not CAPWORDS.match(n.name)]
    out += [(n.lineno, t.id, "variable") for n in tree.body if isinstance(n, ast.Assign) for t in n.targets
            if isinstance(t, ast.Name) and not (UPPER.match(t.id) or SNAKE.match(t.id))]
    return sorted(out)
```

**Cost:** O(n) time, O(v) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the tree; check each kind with if/elif. That costs O(n) time and O(v) space."

"I can do better: A rule table of (node type, regex, rule). That's O(n) time and O(v) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/check-naming-conventions)

<a id="q3"></a>
### Q3. Mutable Default Arguments

🟢 Warm-up · *Bugbear B006*

A default like `items=[]` is created **once** and shared between calls — a classic bug. Return the names of functions with a default that is a list, dict or set literal, or a call to `list()`, `dict()` or `set()`.

**Examples**

- `src = 'def add(item, items=[]):\n    items.append(item)\n    return items\n\ndef ok(item, items=None):\n    items = items or []\n\ndef cfg(options={}, *, tags=set()):\n    pass\n\ndef counter(start=0, seen=dict()):\n    pass\n'` → `['add', 'cfg', 'counter']`

In [ ]:
def mutable_defaults(src):
    """
    A default like items=[] is created once and shared between calls — a classic
    bug. Return the names of functions with a default that is a list, dict or
    set literal, or a call to list(), dict() or set().
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mutable_defaults, [
    (('def add(item, items=[]):\n    items.append(item)\n    return items\n\ndef ok(item, items=None):\n    items = items or []\n\ndef cfg(options={}, *, tags=set()):\n    pass\n\ndef counter(start=0, seen=dict()):\n    pass\n',), ['add', 'cfg', 'counter']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each default with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A helper predicate and any().

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check each default with if-statements.

```python
def mutable_defaults(src):
    out = []
    for fn in ast.walk(ast.parse(src)):
        if not isinstance(fn, ast.FunctionDef):
            continue
        defaults = fn.args.defaults + [d for d in fn.args.kw_defaults if d is not None]
        bad = False
        for d in defaults:
            if isinstance(d, (ast.List, ast.Dict, ast.Set)):
                bad = True
            if isinstance(d, ast.Call) and isinstance(d.func, ast.Name) and d.func.id in ("list", "dict", "set"):
                bad = True
        if bad:
            out.append(fn.name)
    return out
```

**Cost:** O(n) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A helper predicate and any().

```python
def mutable_defaults(src):
    def mutable(d):
        return isinstance(d, (ast.List, ast.Dict, ast.Set)) or (
            isinstance(d, ast.Call) and getattr(d.func, "id", "") in {"list", "dict", "set"})
    return [f.name for f in ast.walk(ast.parse(src)) if isinstance(f, ast.FunctionDef)
            and any(mutable(d) for d in f.args.defaults + f.args.kw_defaults if d is not None)]
```

**Cost:** O(n) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each default with if-statements. That costs O(n) time and O(f) space."

"I can do better: A helper predicate and any(). That's O(n) time and O(f) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `src = 'def f(x=[]):\n    pass\n'`

1. mutable() returns True
2. GeneratorExit raised: 
3. mutable\_defaults() returns ['f']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/mutable-default-arguments)

<a id="q4"></a>
### Q4. Docstring Coverage

🟢 Warm-up · *interrogate*

Count public functions, methods and classes (names not starting with `_`) and return `(coverage % rounded to 1, names missing a docstring in source order)`.

**Examples**

- `src = 'class Cart:\n    """A shopping cart."""\n\n    def add(self, item):\n        """Add an item."""\n\n    def total(self):\n        return 0\n\n    def _recalc(self):\n        pass\n\ndef checkout(cart):\n    pass\n\ndef refund(order):\n    """Refund an order."""\n'` → `(60.0, ['total', 'checkout'])`

In [ ]:
def docstring_coverage(src):
    """
    Count public functions, methods and classes (names not starting with _) and
    return (coverage % rounded to 1, names missing a docstring in source order).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(docstring_coverage, [
    (('class Cart:\n    """A shopping cart."""\n\n    def add(self, item):\n        """Add an item."""\n\n    def total(self):\n        return 0\n\n    def _recalc(self):\n        pass\n\ndef checkout(cart):\n    pass\n\ndef refund(order):\n    """Refund an order."""\n',), (60.0, ['total', 'checkout'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk and check with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Filter public definitions, then use ast.get\_docstring.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk and check with if-statements.

```python
def docstring_coverage(src):
    total, missing = 0, []
    for n in ast.walk(ast.parse(src)):
        if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and not n.name.startswith("_"):
            total += 1
            first = n.body[0]
            has = isinstance(first, ast.Expr) and isinstance(first.value, ast.Constant) and isinstance(first.value.value, str)
            if not has:
                missing.append((n.lineno, n.name))
    missing.sort()
    return (round((total - len(missing)) / total * 100, 1), [m for _, m in missing])
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Filter public definitions, then use ast.get\_docstring.

```python
def docstring_coverage(src):
    defs = sorted((n.lineno, n.name, ast.get_docstring(n)) for n in ast.walk(ast.parse(src))
                  if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and not n.name.startswith("_"))
    missing = [name for _, name, doc in defs if doc is None]
    return (round(100 * (1 - len(missing) / len(defs)), 1), missing)
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk and check with if-statements. That costs O(n) time and O(d) space."

"I can do better: Filter public definitions, then use ast.get\_docstring. That's O(n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/docstring-coverage)

<a id="q5"></a>
### Q5. Type Hint Coverage

🟢 Warm-up · *How much of the code is typed?*

For every function, count its parameters (excluding `self` and `cls`) plus its return value as slots; a slot is covered if it has an annotation. Return `(coverage % rounded to 1, {function: covered/total as 'c/t'})`.

**Examples**

- `src = "def total(prices: list[float], discount: float = 0.0) -> float:\n    pass\n\ndef parse(text, strict=False):\n    pass\n\nclass Repo:\n    def get(self, id: int):\n        pass\n\n    @classmethod\n    def build(cls, url: str) -> 'Repo':\n        pass\n"` → `(60.0, {'total': '3/3', 'parse': '0/3', 'get': '1/2', 'build': '2/2'})`

In [ ]:
def hint_coverage(src):
    """
    For every function, count its parameters (excluding self and cls) plus its
    return value as slots; a slot is covered if it has an annotation. Return
    (coverage % rounded to 1, {function: covered/total as 'c/t'}).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hint_coverage, [
    (("def total(prices: list[float], discount: float = 0.0) -> float:\n    pass\n\ndef parse(text, strict=False):\n    pass\n\nclass Repo:\n    def get(self, id: int):\n        pass\n\n    @classmethod\n    def build(cls, url: str) -> 'Repo':\n        pass\n",), (60.0, {'total': '3/3', 'parse': '0/3', 'get': '1/2', 'build': '2/2'})),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehensions over all argument kinds.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count with loops.

```python
def hint_coverage(src):
    per, cov, tot = {}, 0, 0
    for fn in ast.walk(ast.parse(src)):
        if not isinstance(fn, ast.FunctionDef):
            continue
        args = fn.args.posonlyargs + fn.args.args + fn.args.kwonlyargs
        c = t = 0
        for a in args:
            if a.arg in ("self", "cls"):
                continue
            t += 1
            if a.annotation is not None:
                c += 1
        t += 1
        if fn.returns is not None:
            c += 1
        per[fn.name] = f"{c}/{t}"
        cov += c
        tot += t
    return (round(cov / tot * 100, 1), per)
```

**Cost:** O(n) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehensions over all argument kinds.

```python
def hint_coverage(src):
    per, cov, tot = {}, 0, 0
    for fn in (n for n in ast.walk(ast.parse(src)) if isinstance(n, ast.FunctionDef)):
        slots = [a.annotation for a in fn.args.posonlyargs + fn.args.args + fn.args.kwonlyargs if a.arg not in ("self", "cls")] + [fn.returns]
        c = sum(s is not None for s in slots)
        per[fn.name], cov, tot = f"{c}/{len(slots)}", cov + c, tot + len(slots)
    return (round(100 * cov / tot, 1), per)
```

**Cost:** O(n) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count with loops. That costs O(n) time and O(f) space."

"I can do better: Comprehensions over all argument kinds. That's O(n) time and O(f) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/type-hint-coverage)

<a id="q6"></a>
### Q6. Unreachable Code

🟢 Warm-up · *Statements after return*

Return the line numbers of statements that come after a `return`, `raise`, `break` or `continue` in the same block (only the first unreachable statement of each block).

**Examples**

- `src = "def f(x):\n    if x < 0:\n        raise ValueError('neg')\n        print('never')\n    for i in range(x):\n        if i == 3:\n            break\n            i += 1\n    return x\n    x = 0\n"` → `[4, 8, 10]`

In [ ]:
def unreachable(src):
    """
    Return the line numbers of statements that come after a return, raise, break
    or continue in the same block (only the first unreachable statement of each
    block).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unreachable, [
    (("def f(x):\n    if x < 0:\n        raise ValueError('neg')\n        print('never')\n    for i in range(x):\n        if i == 3:\n            break\n            i += 1\n    return x\n    x = 0\n",), [4, 8, 10]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk every node with a body and scan it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, also covering else blocks via a field list.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk every node with a body and scan it.

```python
def unreachable(src):
    out = []
    for n in ast.walk(ast.parse(src)):
        for field in ("body", "orelse", "finalbody"):
            block = getattr(n, field, None)
            if not isinstance(block, list):
                continue
            for i, s in enumerate(block[:-1]):
                if isinstance(s, (ast.Return, ast.Raise, ast.Break, ast.Continue)):
                    out.append(block[i + 1].lineno)
                    break
    return sorted(out)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, also covering else blocks via a field list.

```python
def unreachable(src):
    STOP = (ast.Return, ast.Raise, ast.Break, ast.Continue)
    out = []
    for n in ast.walk(ast.parse(src)):
        for block in (getattr(n, f, None) for f in ("body", "orelse", "finalbody")):
            if isinstance(block, list):
                i = next((i for i, s in enumerate(block[:-1]) if isinstance(s, STOP)), None)
                if i is not None:
                    out.append(block[i + 1].lineno)
    return sorted(out)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk every node with a body and scan it. That costs O(n) time and O(1) space."

"I can do better: Same, also covering else blocks via a field list. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/unreachable-code)

<a id="q7"></a>
### Q7. Magic Numbers

🟢 Warm-up · *Name your constants*

Inside function bodies, report `(line, value)` for numeric literals other than 0, 1, -1 and 2 — they should be named constants. (Negative numbers appear as a minus applied to a positive literal; report the positive literal.)

**Examples**

- `src = 'TAX = 0.18\n\ndef price(base, qty):\n    if qty > 100:\n        base *= 0.85\n    total = base * qty * (1 + TAX)\n    return round(total, 2)\n\ndef retry():\n    for attempt in range(5):\n        sleep(2 ** attempt * 0.25)\n'` → `[(4, 100), (5, 0.85), (10, 5), (11, 0.25)]`

In [ ]:
def magic_numbers(src):
    """
    Inside function bodies, report (line, value) for numeric literals other than
    0, 1, -1 and 2 — they should be named constants. (Negative numbers appear as
    a minus applied to a positive literal; report the positive literal.)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(magic_numbers, [
    (('TAX = 0.18\n\ndef price(base, qty):\n    if qty > 100:\n        base *= 0.85\n    total = base * qty * (1 + TAX)\n    return round(total, 2)\n\ndef retry():\n    for attempt in range(5):\n        sleep(2 ** attempt * 0.25)\n',), [(4, 100), (5, 0.85), (10, 5), (11, 0.25)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk functions and test each constant.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension over function bodies.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk functions and test each constant.

```python
def magic_numbers(src):
    out = []
    for fn in ast.walk(ast.parse(src)):
        if isinstance(fn, ast.FunctionDef):
            for n in ast.walk(fn):
                if isinstance(n, ast.Constant) and type(n.value) in (int, float) and n.value not in (0, 1, -1, 2):
                    out.append((n.lineno, n.value))
    return sorted(out)
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension over function bodies.

```python
def magic_numbers(src):
    funcs = [f for f in ast.walk(ast.parse(src)) if isinstance(f, ast.FunctionDef)]
    return sorted((n.lineno, n.value) for f in funcs for n in ast.walk(f)
                  if isinstance(n, ast.Constant) and type(n.value) in (int, float) and n.value not in (0, 1, -1, 2))
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk functions and test each constant. That costs O(n) time and O(m) space."

"I can do better: A comprehension over function bodies. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/magic-numbers)

<a id="q8"></a>
### Q8. Functions That Do Too Much

🟢 Warm-up · *Size limits*

Return `(function, reasons)` for functions with more than `max_lines` lines (from the `def` line to its last line) or more than `max_args` parameters (all kinds, excluding `self`). Reasons are `'too long'` and/or `'too many args'`.

**Examples**

- `src = 'def a(x, y):\n    return x + y\n\ndef build(name, size, color, owner, tags, notes):\n    pass\n\ndef big(x):\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    return x\n', max_lines = 25, max_args = 5` → `[('build', ['too many args']), ('big', ['too long'])]`

In [ ]:
def too_big(src, max_lines, max_args):
    """
    Return (function, reasons) for functions with more than max_lines lines
    (from the def line to its last line) or more than max_args parameters (all
    kinds, excluding self). Reasons are 'too long' and/or 'too many args'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(too_big, [
    (('def a(x, y):\n    return x + y\n\ndef build(name, size, color, owner, tags, notes):\n    pass\n\ndef big(x):\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    x += 1\n    return x\n', 25, 5), [('build', ['too many args']), ('big', ['too long'])]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute both measures with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use end\_lineno and list lengths.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute both measures with loops.

```python
def too_big(src, max_lines, max_args):
    out = []
    for fn in ast.walk(ast.parse(src)):
        if not isinstance(fn, ast.FunctionDef):
            continue
        reasons = []
        if fn.end_lineno - fn.lineno + 1 > max_lines:
            reasons.append("too long")
        n = 0
        for a in fn.args.posonlyargs + fn.args.args + fn.args.kwonlyargs:
            if a.arg != "self":
                n += 1
        if fn.args.vararg:
            n += 1
        if fn.args.kwarg:
            n += 1
        if n > max_args:
            reasons.append("too many args")
        if reasons:
            out.append((fn.name, reasons))
    return out
```

**Cost:** O(n) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Use end\_lineno and list lengths.

```python
def too_big(src, max_lines, max_args):
    out = []
    for fn in (f for f in ast.walk(ast.parse(src)) if isinstance(f, ast.FunctionDef)):
        a = fn.args
        n = sum(x.arg != "self" for x in a.posonlyargs + a.args + a.kwonlyargs) + bool(a.vararg) + bool(a.kwarg)
        r = ["too long"] * (fn.end_lineno - fn.lineno + 1 > max_lines) + ["too many args"] * (n > max_args)
        if r:
            out.append((fn.name, r))
    return out
```

**Cost:** O(n) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute both measures with loops. That costs O(n) time and O(f) space."

"I can do better: Use end\_lineno and list lengths. That's O(n) time and O(f) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/functions-that-do-too-much)

<a id="q9"></a>
### Q9. Whitespace Problems

🟢 Warm-up · *W291, W292, W391*

Report `(line, code)`: `'W291'` trailing whitespace on a line, `'W391'` blank line(s) at the end of the file (report the first trailing blank line), `'W292'` no newline at the end of the file (report the last line).

**Examples**

- `src = 'x = 1  \ny = 2\n\n\n'` → `[(1, 'W291'), (3, 'W391')]`
- `src = 'def f():\n\treturn 1 '` → `[(2, 'W291'), (2, 'W292')]`

In [ ]:
def whitespace(src):
    """
    Report (line, code): 'W291' trailing whitespace on a line, 'W391' blank
    line(s) at the end of the file (report the first trailing blank line),
    'W292' no newline at the end of the file (report the last line).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(whitespace, [
    (('x = 1  \ny = 2\n\n\n',), [(1, 'W291'), (3, 'W391')]),
    (('def f():\n\treturn 1 ',), [(2, 'W291'), (2, 'W292')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk lines with flags.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehension plus two end-of-file checks.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk lines with flags.

```python
def whitespace(src):
    lines = src.split("\n")
    has_final_newline = src.endswith("\n")
    if has_final_newline:
        lines = lines[:-1]
    out = []
    for i, l in enumerate(lines, 1):
        if l != l.rstrip():
            out.append((i, "W291"))
    if not has_final_newline:
        out.append((len(lines), "W292"))
    j = len(lines)
    while j > 0 and lines[j - 1].strip() == "":
        j -= 1
    if j < len(lines):
        out.append((j + 1, "W391"))
    return sorted(out)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehension plus two end-of-file checks.

```python
def whitespace(src):
    lines = src.splitlines()
    out = [(i, "W291") for i, l in enumerate(lines, 1) if l and l != l.rstrip()]
    if not src.endswith("\n"):
        out.append((len(lines), "W292"))
    tail = len(lines) - len(src.rstrip().splitlines()) if src.strip() else 0
    if tail:
        out.append((len(lines) - tail + 1, "W391"))
    return sorted(out)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk lines with flags. That costs O(n) time and O(n) space."

"I can do better: Comprehension plus two end-of-file checks. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/whitespace-problems)

<a id="q10"></a>
### Q10. Shadowed Builtins

🟢 Warm-up · *Don't name a variable list*

Return sorted `(line, name)` for assignments, function parameters and function names that reuse a Python builtin name (`list`, `id`, `type`, `sum`, …). Shadowing breaks the builtin for the rest of the scope.

**Examples**

- `src = "def max(values):\n    return sorted(values)[-1]\n\ndef load(id, type='json'):\n    list = []\n    sum = 0\n    total = 0\n    return list\n"` → `[(1, 'max'), (4, 'id'), (4, 'type'), (5, 'list'), (6, 'sum')]`

In [ ]:
import builtins

def shadowed(src):
    """
    Return sorted (line, name) for assignments, function parameters and function
    names that reuse a Python builtin name (list, id, type, sum, …). Shadowing
    breaks the builtin for the rest of the scope.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shadowed, [
    (("def max(values):\n    return sorted(values)[-1]\n\ndef load(id, type='json'):\n    list = []\n    sum = 0\n    total = 0\n    return list\n",), [(1, 'max'), (4, 'id'), (4, 'type'), (5, 'list'), (6, 'sum')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each node kind against the builtins with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One generator of (line, name) for all binding nodes.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check each node kind against the builtins with if-statements.

```python
def shadowed(src):
    names = set(dir(builtins))
    out = []
    for n in ast.walk(ast.parse(src)):
        if isinstance(n, ast.FunctionDef) and n.name in names:
            out.append((n.lineno, n.name))
        if isinstance(n, ast.arg) and n.arg in names:
            out.append((n.lineno, n.arg))
        if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store) and n.id in names:
            out.append((n.lineno, n.id))
    return sorted(out)
```

**Cost:** O(n) time, O(v) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One generator of (line, name) for all binding nodes.

```python
def shadowed(src):
    def bound(n):
        if isinstance(n, ast.FunctionDef):
            yield n.lineno, n.name
        elif isinstance(n, ast.arg):
            yield n.lineno, n.arg
        elif isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store):
            yield n.lineno, n.id
    b = set(dir(builtins))
    return sorted(x for n in ast.walk(ast.parse(src)) for x in bound(n) if x[1] in b)
```

**Cost:** O(n) time, O(v) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each node kind against the builtins with if-statements. That costs O(n) time and O(v) space."

"I can do better: One generator of (line, name) for all binding nodes. That's O(n) time and O(v) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `src = 'dict = {}\n'`

1. bound() returns None
2. bound() returns None
3. bound() returns (1, 'dict')
4. bound() returns None
5. bound() returns None
6. bound() returns None
7. shadowed() returns [(1, 'dict')]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/shadowed-builtins)

<a id="q11"></a>
### Q11. Old-Style String Formatting

🟢 Warm-up · *Use f-strings*

Report `(line, style)` for `'%'` formatting (a string literal on the left of `%`) and `'format'` (calling `.format` on a string literal). Tools like ruff and pyupgrade rewrite these as f-strings.

**Examples**

- `src = "name = 'Asha'\nmsg = 'Hello %s, you have %d items' % (name, 3)\nline = '{} -> {}'.format(a, b)\nok = f'{name} logged in'\nn = 10 % 3\n"` → `[(2, '%'), (3, 'format')]`

In [ ]:
def old_format(src):
    """
    Report (line, style) for '%' formatting (a string literal on the left of %)
    and 'format' (calling .format on a string literal). Tools like ruff and
    pyupgrade rewrite these as f-strings.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(old_format, [
    (("name = 'Asha'\nmsg = 'Hello %s, you have %d items' % (name, 3)\nline = '{} -> {}'.format(a, b)\nok = f'{name} logged in'\nn = 10 % 3\n",), [(2, '%'), (3, 'format')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the tree and check node shapes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk the tree and check node shapes.

```python
def old_format(src):
    out = []
    for n in ast.walk(ast.parse(src)):
        if isinstance(n, ast.BinOp) and isinstance(n.op, ast.Mod) and isinstance(n.left, ast.Constant) and isinstance(n.left.value, str):
            out.append((n.lineno, "%"))
        if isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute) and n.func.attr == "format" \
                and isinstance(n.func.value, ast.Constant) and isinstance(n.func.value.value, str):
            out.append((n.lineno, "format"))
    return sorted(out)
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two comprehensions.

```python
def old_format(src):
    nodes = list(ast.walk(ast.parse(src)))
    is_str = lambda x: isinstance(x, ast.Constant) and isinstance(x.value, str)
    return sorted([(n.lineno, "%") for n in nodes if isinstance(n, ast.BinOp) and isinstance(n.op, ast.Mod) and is_str(n.left)]
                  + [(n.lineno, "format") for n in nodes if isinstance(n, ast.Call) and getattr(n.func, "attr", "") == "format" and is_str(getattr(n.func, "value", None))])
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the tree and check node shapes. That costs O(n) time and O(m) space."

"I can do better: Two comprehensions. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/old-style-string-formatting)

<a id="q12"></a>
### Q12. Count Lines of Code

🟢 Warm-up · *cloc*

Return `{'code': …, 'comment': …, 'blank': …}`: blank lines are empty after stripping; comment lines start with `#` after stripping; everything else is code.

**Examples**

- `src = "# app entry point\nimport os\n\ndef main():\n    # read config\n    path = os.getenv('CFG')  # inline comments count as code\n\n    return path\n"` → `{'code': 4, 'comment': 2, 'blank': 2}`

In [ ]:
def cloc(src):
    """
    Return {'code': …, 'comment': …, 'blank': …}: blank lines are empty after
    stripping; comment lines start with # after stripping; everything else is
    code.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cloc, [
    (("# app entry point\nimport os\n\ndef main():\n    # read config\n    path = os.getenv('CFG')  # inline comments count as code\n\n    return path\n",), {'code': 4, 'comment': 2, 'blank': 2}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Counters in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter` over a classifier.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Counters in a loop.

```python
def cloc(src):
    code = comment = blank = 0
    for line in src.splitlines():
        s = line.strip()
        if not s:
            blank += 1
        elif s.startswith("#"):
            comment += 1
        else:
            code += 1
    return {"code": code, "comment": comment, "blank": blank}
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter` over a classifier.

```python
def cloc(src):
    c = Counter("blank" if not l.strip() else "comment" if l.strip()[0] == "#" else "code" for l in src.splitlines())
    return {k: c[k] for k in ("code", "comment", "blank")}
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Counters in a loop. That costs O(n) time and O(1) space."

"I can do better: `Counter` over a classifier. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/count-lines-of-code)

<a id="q13"></a>
### Q13. Find Unused Imports

🟡 Interview standard · *Ruff F401*

Parse the source and return the imported names (the name bound in the module, e.g. `np` for `import numpy as np`, `os` for `import os.path`) that are never used, sorted.

**Examples**

- `src = 'import os\nimport sys\nimport numpy as np\nfrom pathlib import Path, PurePath\nfrom typing import List\nimport os.path\n\ndef main(args: List[str]) -> None:\n    p = Path(args[0])\n    print(os.getcwd(), p)\n'` → `['PurePath', 'np', 'sys']`

In [ ]:
def unused_imports(src):
    """
    Parse the source and return the imported names (the name bound in the
    module, e.g. np for import numpy as np, os for import os.path) that are
    never used, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unused_imports, [
    (('import os\nimport sys\nimport numpy as np\nfrom pathlib import Path, PurePath\nfrom typing import List\nimport os.path\n\ndef main(args: List[str]) -> None:\n    p = Path(args[0])\n    print(os.getcwd(), p)\n',), ['PurePath', 'np', 'sys']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect imports, then scan every node for each name.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Collect used names once with the helper.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Collect imports, then scan every node for each name.

```python
def unused_imports(src):
    tree = ast.parse(src)
    imported = []
    for n in ast.walk(tree):
        if isinstance(n, ast.Import):
            for a in n.names:
                imported.append(a.asname or a.name.split(".")[0])
        elif isinstance(n, ast.ImportFrom):
            for a in n.names:
                imported.append(a.asname or a.name)
    unused = []
    for name in imported:
        found = False
        for n in ast.walk(tree):
            if isinstance(n, ast.Name) and n.id == name:
                found = True
        if not found and name not in unused:
            unused.append(name)
    return sorted(unused)
```

**Cost:** O(i · n) time, O(i) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Collect used names once with the helper.

```python
def names_used(tree):
    used = set()
    for n in ast.walk(tree):
        if isinstance(n, ast.Name):
            used.add(n.id)
        elif isinstance(n, ast.Attribute):
            base = n
            while isinstance(base, ast.Attribute):
                base = base.value
            if isinstance(base, ast.Name):
                used.add(base.id)
    return used


def unused_imports(src):
    tree = ast.parse(src)
    bound = {a.asname or (a.name.split(".")[0] if isinstance(n, ast.Import) else a.name)
             for n in ast.walk(tree) if isinstance(n, (ast.Import, ast.ImportFrom)) for a in n.names}
    return sorted(bound - names_used(tree))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect imports, then scan every node for each name. That costs O(i · n) time and O(i) space."

"I can do better: Collect used names once with the helper. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/find-unused-imports)

<a id="q14"></a>
### Q14. Cyclomatic Complexity

🟡 Interview standard · *How many paths through a function?*

Complexity = 1 + the number of decision points: each `if`/`elif`, `for`, `while`, `except` handler, conditional expression, comprehension `if`, and each extra operand of `and`/`or`. Return `{function name: complexity}`. Above ~10, a function is hard to test.

**Examples**

- `src = "def ship(order):\n    if not order.items:\n        return None\n    for item in order.items:\n        if item.weight > 30 and not item.freight_ok:\n            raise ValueError('too heavy')\n    try:\n        rate = lookup(order.zone)\n    except KeyError:\n        rate = 0\n    return rate if rate else default_rate()\n\ndef tiny(x):\n    return [y for y in x if y > 0]\n"` → `{'ship': 7, 'tiny': 2}`

In [ ]:
def complexity(src):
    """
    Complexity = 1 + the number of decision points: each if/elif, for, while,
    except handler, conditional expression, comprehension if, and each extra
    operand of and/or. Return {function name: complexity}. Above ~10, a function
    is hard to test.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(complexity, [
    (("def ship(order):\n    if not order.items:\n        return None\n    for item in order.items:\n        if item.weight > 30 and not item.freight_ok:\n            raise ValueError('too heavy')\n    try:\n        rate = lookup(order.zone)\n    except KeyError:\n        rate = 0\n    return rate if rate else default_rate()\n\ndef tiny(x):\n    return [y for y in x if y > 0]\n",), {'ship': 7, 'tiny': 2}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the function and count each node type with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A tuple of decision node types and a sum.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk the function and count each node type with if-statements.

```python
def complexity(src):
    out = {}
    for fn in ast.parse(src).body:
        if not isinstance(fn, ast.FunctionDef):
            continue
        c = 1
        for n in ast.walk(fn):
            if isinstance(n, (ast.If, ast.For, ast.While, ast.ExceptHandler, ast.IfExp)):
                c += 1
            elif isinstance(n, ast.BoolOp):
                c += len(n.values) - 1
            elif isinstance(n, ast.comprehension):
                c += len(n.ifs)
        out[fn.name] = c
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A tuple of decision node types and a sum.

```python
def complexity(src):
    DECISIONS = (ast.If, ast.For, ast.While, ast.ExceptHandler, ast.IfExp)
    def score(n):
        if isinstance(n, DECISIONS):
            return 1
        if isinstance(n, ast.BoolOp):
            return len(n.values) - 1
        return len(n.ifs) if isinstance(n, ast.comprehension) else 0
    return {f.name: 1 + sum(map(score, ast.walk(f))) for f in ast.parse(src).body if isinstance(f, ast.FunctionDef)}
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the function and count each node type with if-statements. That costs O(n) time and O(1) space."

"I can do better: A tuple of decision node types and a sum. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `src = 'def f(x):\n    if x:\n        return 1\n    return 0\n'`

1. score() returns 0
2. score() returns 0
3. score() returns 1
4. score() returns 0
5. score() returns 0
6. score() returns 0
7. score() returns 0
8. score() returns 0
9. score() returns 0
10. score() returns 0
11. complexity() returns {'f': 2}

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/cyclomatic-complexity)

<a id="q15"></a>
### Q15. Swallowed Exceptions

🟡 Interview standard · *Errors that vanish*

Return `(line, issue)` for each `except` handler: `'bare except'` if it has no exception type, `'swallowed'` if its body is only `pass` (or `...`), or both (sorted) as separate entries. Silent failures are the hardest bugs to find in production.

**Examples**

- `src = "try:\n    load()\nexcept:\n    pass\n\ntry:\n    parse()\nexcept ValueError:\n    log.warning('bad input')\n\ntry:\n    send()\nexcept Exception:\n    pass\n\ntry:\n    go()\nexcept:\n    retry()\n"` → `[(3, 'bare except'), (3, 'swallowed'), (13, 'swallowed'), (18, 'bare except')]`

In [ ]:
def swallowed(src):
    """
    Return (line, issue) for each except handler: 'bare except' if it has no
    exception type, 'swallowed' if its body is only pass (or ...), or both
    (sorted) as separate entries. Silent failures are the hardest bugs to find
    in production.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(swallowed, [
    (("try:\n    load()\nexcept:\n    pass\n\ntry:\n    parse()\nexcept ValueError:\n    log.warning('bad input')\n\ntry:\n    send()\nexcept Exception:\n    pass\n\ntry:\n    go()\nexcept:\n    retry()\n",), [(3, 'bare except'), (3, 'swallowed'), (13, 'swallowed'), (18, 'bare except')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk handlers with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension over handlers and checks.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk handlers with if-statements.

```python
def swallowed(src):
    out = []
    for n in ast.walk(ast.parse(src)):
        if isinstance(n, ast.ExceptHandler):
            if n.type is None:
                out.append((n.lineno, "bare except"))
            only_pass = True
            for s in n.body:
                if not (isinstance(s, ast.Pass) or (isinstance(s, ast.Expr) and isinstance(s.value, ast.Constant) and s.value.value is Ellipsis)):
                    only_pass = False
            if only_pass:
                out.append((n.lineno, "swallowed"))
    return sorted(out)
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension over handlers and checks.

```python
def swallowed(src):
    def empty(s):
        return isinstance(s, ast.Pass) or (isinstance(s, ast.Expr) and getattr(s.value, "value", None) is Ellipsis)
    hs = [n for n in ast.walk(ast.parse(src)) if isinstance(n, ast.ExceptHandler)]
    return sorted([(h.lineno, "bare except") for h in hs if h.type is None] + [(h.lineno, "swallowed") for h in hs if all(map(empty, h.body))])
```

**Cost:** O(n) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk handlers with if-statements. That costs O(n) time and O(h) space."

"I can do better: A comprehension over handlers and checks. That's O(n) time and O(h) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/swallowed-exceptions)

<a id="q16"></a>
### Q16. Check Call Arguments Against Hints

🟡 Interview standard · *What mypy and pydantic catch*

Given parameter hints (strings like `'int'`, `'float'`, `'str'`, `'bool'`, `'list[int]'`, `'str | None'`) and a list of calls (dicts of argument values), return for each call the sorted parameter names whose value doesn't match. An int is acceptable for float; a bool is not an int. Use `type_ok`.

**Examples**

- `hints = {'user_id': 'int', 'amount': 'float', 'note': 'str | None', 'tags': 'list[str]', 'urgent': 'bool'}, calls = [{'user_id': 7, 'amount': 10, 'note': None, 'tags': ['a'], 'urgent': False}, {'user_id': '7', 'amount': 9.5, 'note': 'hi', 'tags': [], 'urgent': 1}, {'user_id': True, 'amount': '3', 'note': 5, 'tags': ['a', 2], 'urgent': True}]` → `[[], ['urgent', 'user_id'], ['amount', 'note', 'tags', 'user_id']]`

In [ ]:
def bad_args(hints, calls):
    """
    Given parameter hints (strings like 'int', 'float', 'str', 'bool',
    'list[int]', 'str | None') and a list of calls (dicts of argument values),
    return for each call the sorted parameter names whose value doesn't match.
    An int is acceptable for float; a bool is not an int. Use type_ok.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bad_args, [
    (({'user_id': 'int', 'amount': 'float', 'note': 'str | None', 'tags': 'list[str]', 'urgent': 'bool'}, [{'user_id': 7, 'amount': 10, 'note': None, 'tags': ['a'], 'urgent': False}, {'user_id': '7', 'amount': 9.5, 'note': 'hi', 'tags': [], 'urgent': 1}, {'user_id': True, 'amount': '3', 'note': 5, 'tags': ['a', 2], 'urgent': True}]), [[], ['urgent', 'user_id'], ['amount', 'note', 'tags', 'user_id']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over calls and parameters.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension per call.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over calls and parameters.

```python
def type_ok(value, hint):
    if hint.endswith(" | None"):
        return value is None or type_ok(value, hint[:-7])
    if hint.startswith("list[") and hint.endswith("]"):
        return isinstance(value, list) and all(type_ok(v, hint[5:-1]) for v in value)
    if hint == "float":
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    if hint == "int":
        return isinstance(value, int) and not isinstance(value, bool)
    return type(value).__name__ == hint


def bad_args(hints, calls):
    out = []
    for call in calls:
        wrong = []
        for name, hint in hints.items():
            if not type_ok(call[name], hint):
                wrong.append(name)
        out.append(sorted(wrong))
    return out
```

**Cost:** O(c · p) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension per call.

```python
def type_ok(value, hint):
    if hint.endswith(" | None"):
        return value is None or type_ok(value, hint[:-7])
    if hint.startswith("list[") and hint.endswith("]"):
        return isinstance(value, list) and all(type_ok(v, hint[5:-1]) for v in value)
    if hint == "float":
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    if hint == "int":
        return isinstance(value, int) and not isinstance(value, bool)
    return type(value).__name__ == hint


def bad_args(hints, calls):
    return [sorted(n for n, h in hints.items() if not type_ok(c[n], h)) for c in calls]
```

**Cost:** O(c · p) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over calls and parameters. That costs O(c · p) time and O(p) space."

"I can do better: A comprehension per call. That's O(c · p) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/check-call-arguments-against-hints)

<a id="q17"></a>
### Q17. Sort Imports Like isort

🟡 Interview standard · *Three groups, alphabetical*

Order import lines into three groups — standard library (top module in `STDLIB`), third-party, then local (top module equals `local`) — separated by one blank line (`''`). Within a group, plain `import x` lines come before `from x import y`, each sorted case-insensitively by module.

**Examples**

- `lines = ['from app.models import User', 'import requests', 'import os', 'from pathlib import Path', 'import numpy as np', 'from typing import Any', 'import app.config', 'import json', 'from fastapi import FastAPI'], local = 'app'` → `['import json', 'import os', 'from pathlib import Path', 'from typing import Any', '', 'import numpy as np', 'import requests', 'from fastapi import FastAPI', '', 'import app.config', 'from app.models import User']`

In [ ]:
STDLIB = {"os", "sys", "json", "re", "math", "pathlib", "typing", "datetime", "collections", "logging", "itertools", "functools"}

def sort_imports(lines, local):
    """
    Order import lines into three groups — standard library (top module in
    STDLIB), third-party, then local (top module equals local) — separated by
    one blank line (''). Within a group, plain import x lines come before from x
    import y, each sorted case-insensitively by module.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sort_imports, [
    ((['from app.models import User', 'import requests', 'import os', 'from pathlib import Path', 'import numpy as np', 'from typing import Any', 'import app.config', 'import json', 'from fastapi import FastAPI'], 'app'), ['import json', 'import os', 'from pathlib import Path', 'from typing import Any', '', 'import numpy as np', 'import requests', 'from fastapi import FastAPI', '', 'import app.config', 'from app.models import User']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Bucket lines with loops, then sort each bucket.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A sort key of (group, style, module).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Bucket lines with loops, then sort each bucket.

```python
def sort_imports(lines, local):
    groups = [[], [], []]
    for line in lines:
        module = line.split()[1]
        top = module.split(".")[0]
        g = 0 if top in STDLIB else 2 if top == local else 1
        groups[g].append(line)
    out = []
    for g in groups:
        if not g:
            continue
        plain = sorted([l for l in g if l.startswith("import ")], key=lambda l: l.split()[1].lower())
        froms = sorted([l for l in g if l.startswith("from ")], key=lambda l: l.split()[1].lower())
        if out:
            out.append("")
        out += plain + froms
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A sort key of (group, style, module).

```python
def sort_imports(lines, local):
    def key(line):
        mod = line.split()[1]
        top = mod.split(".")[0]
        return (0 if top in STDLIB else 2 if top == local else 1, line.startswith("from"), mod.lower())
    out, prev = [], None
    for line in sorted(lines, key=key):
        if prev is not None and key(line)[0] != prev:
            out.append("")
        out.append(line)
        prev = key(line)[0]
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Bucket lines with loops, then sort each bucket. That costs O(n log n) time and O(n) space."

"I can do better: A sort key of (group, style, module). That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `lines = ['import sys', 'import requests'], local = 'app'`

1. key() returns (0, False, 'sys')
2. key() returns (1, False, 'requests')
3. key() returns (0, False, 'sys')
4. key() returns (1, False, 'requests')
5. key() returns (1, False, 'requests')
6. sort\_imports() returns ['import sys', '', 'import requests']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/sort-imports-like-isort)

<a id="q18"></a>
### Q18. Find Duplicated Code

🟡 Interview standard · *Copy-paste detection*

Normalise each line (strip spaces; skip blank lines) and find every block of `k` consecutive normalised lines that appears in more than one place. Return sorted `(file, start line)` pairs of the duplicated blocks' locations (line numbers are the original 1-based numbers).

**Examples**

- `files = {'a.py': 'def f(x):\n    y = x * 2\n    z = y + 1\n    return z\n', 'b.py': 'def g(x):\n\n    y = x * 2\n    z = y + 1\n    return z\n'}, k = 3` → `[('a.py', 2), ('b.py', 3)]`

In [ ]:
def duplicates(files, k):
    """
    Normalise each line (strip spaces; skip blank lines) and find every block of
    k consecutive normalised lines that appears in more than one place. Return
    sorted (file, start line) pairs of the duplicated blocks' locations (line
    numbers are the original 1-based numbers).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(duplicates, [
    (({'a.py': 'def f(x):\n    y = x * 2\n    z = y + 1\n    return z\n', 'b.py': 'def g(x):\n\n    y = x * 2\n    z = y + 1\n    return z\n'}, 3), [('a.py', 2), ('b.py', 3)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare every block with every other block.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Index blocks in a dict keyed by their text.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare every block with every other block.

```python
def blocks(files, k):
    out = []
    for name, src in files.items():
        lines = [(i, l.strip()) for i, l in enumerate(src.splitlines(), 1) if l.strip()]
        for j in range(len(lines) - k + 1):
            out.append((name, lines[j][0], tuple(t for _, t in lines[j:j + k])))
    return out


def duplicates(files, k):
    bl = blocks(files, k)
    found = set()
    for i in range(len(bl)):
        for j in range(len(bl)):
            if i != j and bl[i][2] == bl[j][2]:
                found.add((bl[i][0], bl[i][1]))
    return sorted(found)
```

**Cost:** O(b²) time, O(b) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Index blocks in a dict keyed by their text.

```python
def blocks(files, k):
    out = []
    for name, src in files.items():
        lines = [(i, l.strip()) for i, l in enumerate(src.splitlines(), 1) if l.strip()]
        for j in range(len(lines) - k + 1):
            out.append((name, lines[j][0], tuple(t for _, t in lines[j:j + k])))
    return out


def duplicates(files, k):
    where = defaultdict(list)
    for name, line, text in blocks(files, k):
        where[text].append((name, line))
    return sorted(loc for locs in where.values() if len(locs) > 1 for loc in locs)
```

**Cost:** O(b) time, O(b) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare every block with every other block. That costs O(b²) time and O(b) space."

"I can do better: Index blocks in a dict keyed by their text. That's O(b) time and O(b) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/find-duplicated-code)

<a id="q19"></a>
### Q19. Enforce Architecture Layers

🟡 Interview standard · *import-linter*

Layers are listed from top to bottom (e.g. `api → services → repositories → models`). A module may import from lower layers but never from a higher one. `imports` maps modules (named `layer.something`) to the modules they import. Return sorted violations `(importer, imported)`.

**Examples**

- `layers = ['api', 'services', 'repositories', 'models'], imports = {'api.orders': ['services.orders', 'models.order'], 'services.orders': ['repositories.orders', 'api.schemas'], 'repositories.orders': ['models.order', 'services.pricing'], 'models.order': [], 'services.pricing': ['models.order']}` → `[('repositories.orders', 'services.pricing'), ('services.orders', 'api.schemas')]`

In [ ]:
def layer_violations(layers, imports):
    """
    Layers are listed from top to bottom (e.g. api → services → repositories →
    models). A module may import from lower layers but never from a higher one.
    imports maps modules (named layer.something) to the modules they import.
    Return sorted violations (importer, imported).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(layer_violations, [
    ((['api', 'services', 'repositories', 'models'], {'api.orders': ['services.orders', 'models.order'], 'services.orders': ['repositories.orders', 'api.schemas'], 'repositories.orders': ['models.order', 'services.pricing'], 'models.order': [], 'services.pricing': ['models.order']}), [('repositories.orders', 'services.pricing'), ('services.orders', 'api.schemas')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Look up both layers' positions with list.index each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict from layer to rank.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Look up both layers' positions with list.index each time.

```python
def layer_violations(layers, imports):
    out = []
    for mod, deps in imports.items():
        for d in deps:
            if layers.index(d.split(".")[0]) < layers.index(mod.split(".")[0]):
                out.append((mod, d))
    return sorted(out)
```

**Cost:** O(e · l) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict from layer to rank.

```python
def layer_violations(layers, imports):
    rank = {l: i for i, l in enumerate(layers)}
    top = lambda m: rank[m.split(".")[0]]
    return sorted((m, d) for m, deps in imports.items() for d in deps if top(d) < top(m))
```

**Cost:** O(e) time, O(l) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Look up both layers' positions with list.index each time. That costs O(e · l) time and O(1) space."

"I can do better: A dict from layer to rank. That's O(e) time and O(l) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/enforce-architecture-layers)

<a id="q20"></a>
### Q20. Which Lint Rules Are On?

🟡 Interview standard · *select and ignore*

Ruff enables rules by code prefix: `select` turns prefixes on, `ignore` turns them off, and the **longest** matching prefix decides (if a select and an ignore prefix are equally long, ignore wins). Return the enabled rule codes from `rules`, in order.

**Examples**

- `select = ['E', 'F', 'I', 'B006', 'UP'], ignore = ['E501', 'F4', 'UP0'], rules = ['E501', 'E711', 'F401', 'F841', 'I001', 'B006', 'B008', 'UP006', 'UP101', 'W291']` → `['E711', 'F841', 'I001', 'B006', 'UP101']`

In [ ]:
def enabled_rules(select, ignore, rules):
    """
    Ruff enables rules by code prefix: select turns prefixes on, ignore turns
    them off, and the longest matching prefix decides (if a select and an ignore
    prefix are equally long, ignore wins). Return the enabled rule codes from
    rules, in order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(enabled_rules, [
    ((['E', 'F', 'I', 'B006', 'UP'], ['E501', 'F4', 'UP0'], ['E501', 'E711', 'F401', 'F841', 'I001', 'B006', 'B008', 'UP006', 'UP101', 'W291']), ['E711', 'F841', 'I001', 'B006', 'UP101']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Find the longest matching prefix in each list by looping.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

max() over the matching prefixes with a key.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Find the longest matching prefix in each list by looping.

```python
def enabled_rules(select, ignore, rules):
    out = []
    for r in rules:
        best_sel = 0
        for p in select:
            if r.startswith(p) and len(p) > best_sel:
                best_sel = len(p)
        best_ign = 0
        for p in ignore:
            if r.startswith(p) and len(p) > best_ign:
                best_ign = len(p)
        if best_sel > best_ign:
            out.append(r)
    return out
```

**Cost:** O(r · p) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** max() over the matching prefixes with a key.

```python
def enabled_rules(select, ignore, rules):
    longest = lambda prefixes, r: max((len(p) for p in prefixes if r.startswith(p)), default=0)
    return [r for r in rules if longest(select, r) > longest(ignore, r)]
```

**Cost:** O(r · p) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Find the longest matching prefix in each list by looping. That costs O(r · p) time and O(1) space."

"I can do better: max() over the matching prefixes with a key. That's O(r · p) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/code-quality/which-lint-rules-are-on)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why use a formatter like Black or ruff format?</b></summary>

It removes style debates and noisy diffs: everyone's code looks the same, and reviews focus on logic.

</details>

<details>
<summary><b>2. Do type hints make Python slower?</b></summary>

No. They're not enforced at runtime; they're read by type checkers and editors. Use pydantic or explicit checks where runtime validation matters.

</details>

<details>
<summary><b>3. What's wrong with a mutable default argument?</b></summary>

The default object is created once when the function is defined and shared by every call, so changes leak between calls. Use `None` and create the list inside.

</details>

<details>
<summary><b>4. What is cyclomatic complexity?</b></summary>

The number of independent paths through a function: 1 plus the number of decision points. It's roughly the minimum number of tests needed to cover every branch.

</details>

<details>
<summary><b>5. Why is a bare except bad?</b></summary>

It also catches KeyboardInterrupt and SystemExit and hides bugs. Catch the specific exceptions you can handle, and log or re-raise the rest.

</details>

<details>
<summary><b>6. How do you introduce linting to an old codebase?</b></summary>

Add the formatter in one commit, enable a small rule set, auto-fix, then add rules gradually; lint only changed files in CI at first if the backlog is large.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why code quality tools exist
- [ ] I can explain: How a linter reads your code
- [ ] I can explain: Type hints and type checking
- [ ] I can explain: Keeping a codebase healthy

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Check Call Arguments Against Hints
- [ ] [Q17](#q17) Sort Imports Like isort
- [ ] [Q18](#q18) Find Duplicated Code
- [ ] [Q19](#q19) Enforce Architecture Layers
- [ ] [Q20](#q20) Which Lint Rules Are On?

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.